Ниже рекорд - 7,530%

In [ ]:
import pandas as pd
import numpy as np
import requests
import lightgbm as lgb
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit
import optuna
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# КОНФИГ
# =============================================================================
TRAIN_PATH = 'train_dataset.csv'
VALID_PATH = 'valid_features.csv'
OUTPUT_VALID = 'solverdata.csv'

TARGET = 'Выработка. Результирующий расчет'
DT_COL = 'METEOFORECASTHOUR_OPENM_Datetime'

CAPACITY    = 90.09
N_TURBINES  = 26
LATITUDE    = 46.8268455973
LONGITUDE   = 38.7179393185

SEEDS_LGB = [42, 123, 777, 2024, 9001]
SEEDS_CAT = [42, 123, 777]          # блендинг сидов для CatBoost
LGB_ITER_MULTIPLIER = 1.1
N_WAKE_SECTORS = 12

# =============================================================================
# 1. ЗАГРУЗКА ИСХОДНЫХ ДАННЫХ
# =============================================================================
train = pd.read_csv(TRAIN_PATH)
valid = pd.read_csv(VALID_PATH)
train[DT_COL] = pd.to_datetime(train[DT_COL])
valid[DT_COL] = pd.to_datetime(valid[DT_COL])
valid_order = valid[DT_COL].copy()
train = train.sort_values(DT_COL).reset_index(drop=True)

# =============================================================================
# 2. ЗАГРУЗКА ERA5
# =============================================================================
def fetch_era5(start_date, end_date):
    url = "https://archive-api.open-meteo.com/v1/era5"
    params = {
        "latitude": LATITUDE, "longitude": LONGITUDE,
        "start_date": start_date, "end_date": end_date,
        "hourly": [
            "wind_speed_10m", "wind_speed_100m",
            "wind_direction_10m", "wind_direction_100m",
            "temperature_2m", "pressure_msl"
        ],
        "timezone": "UTC"
    }
    r = requests.get(url, params=params).json()["hourly"]
    return pd.DataFrame({
        DT_COL: pd.to_datetime(r["time"]),
        "ws10_era5": r["wind_speed_10m"],
        "ws100_era5": r["wind_speed_100m"],
        "wd10_era5": r["wind_direction_10m"],   # градусы
        "wd100_era5": r["wind_direction_100m"], # градусы
        "temp2m_era5": r["temperature_2m"],
        "pressure_era5": r["pressure_msl"],
    })

print("Загрузка ERA5...")
train_era5 = fetch_era5(train[DT_COL].min().strftime('%Y-%m-%d'),
                        train[DT_COL].max().strftime('%Y-%m-%d'))
valid_era5 = fetch_era5(valid[DT_COL].min().strftime('%Y-%m-%d'),
                        valid[DT_COL].max().strftime('%Y-%m-%d'))

# =============================================================================
# 3. ЗАГРУЗКА NASA POWER (расширенный набор)
# =============================================================================
def fetch_nasa_power(start_date, end_date):
    url = "https://power.larc.nasa.gov/api/temporal/hourly/point"
    params = {
        "parameters": "WS10M,WS50M,WS2M,RH2M,T2M,PS,QV2M,PRECTOTCORR",
        "community": "RE",
        "longitude": LONGITUDE,
        "latitude": LATITUDE,
        "start": start_date.replace('-', ''),
        "end": end_date.replace('-', ''),
        "format": "JSON"
    }
    r = requests.get(url, params=params)
    r.raise_for_status()
    data = r.json()
    rec = data["properties"]["parameter"]
    timestamps = sorted(rec["WS10M"].keys())
    return pd.DataFrame({
        DT_COL: pd.to_datetime(timestamps, format='%Y%m%d%H'),
        "ws10_power": [rec["WS10M"][t] for t in timestamps],
        "ws50_power": [rec["WS50M"][t] for t in timestamps],
        "ws2_power":  [rec["WS2M"][t] for t in timestamps],
        "rh2m_power": [rec["RH2M"][t] for t in timestamps],
        "temp2m_power": [rec["T2M"][t] for t in timestamps],
        "pressure_power": [rec["PS"][t] for t in timestamps],
        "qv2m_power": [rec["QV2M"][t] for t in timestamps],
        "precip_power": [rec["PRECTOTCORR"][t] for t in timestamps],
    })

print("Загрузка NASA POWER...")
train_power = fetch_nasa_power(train[DT_COL].min().strftime('%Y-%m-%d'),
                               train[DT_COL].max().strftime('%Y-%m-%d'))
valid_power = fetch_nasa_power(valid[DT_COL].min().strftime('%Y-%m-%d'),
                               valid[DT_COL].max().strftime('%Y-%m-%d'))

# =============================================================================
# 4. ЗАГРУЗКА AOD (Open-Meteo Air Quality API)
# =============================================================================
def fetch_aod(start_date, end_date):
    """Загружает аэрозольную оптическую толщину (AOD) через Open-Meteo Air Quality."""
    url = "https://air-quality-api.open-meteo.com/v1/air-quality"
    params = {
        "latitude": LATITUDE,
        "longitude": LONGITUDE,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": "dust",
        "timezone": "UTC"
    }
    try:
        r = requests.get(url, params=params, timeout=10)
        r.raise_for_status()
        data = r.json()
        if "hourly" in data and "time" in data["hourly"]:
            return pd.DataFrame({
                DT_COL: pd.to_datetime(data["hourly"]["time"]),
                "aod": data["hourly"].get("dust", None)  # dust AOD
            })
        else:
            print("⚠️ AOD API: пустой ответ")
            return pd.DataFrame(columns=[DT_COL, "aod"])
    except Exception as e:
        print(f"⚠️ AOD не загружен: {e}")
        return pd.DataFrame(columns=[DT_COL, "aod"])

print("Загрузка AOD...")
train_aod = fetch_aod(train[DT_COL].min().strftime('%Y-%m-%d'),
                      train[DT_COL].max().strftime('%Y-%m-%d'))
valid_aod = fetch_aod(valid[DT_COL].min().strftime('%Y-%m-%d'),
                      valid[DT_COL].max().strftime('%Y-%m-%d'))

# =============================================================================
# 5. РАСШИРЕННЫЕ ДАННЫЕ РЕЛЬЕФА (статистики окрестности)
# =============================================================================
def get_elevation(lat, lon):
    url = f"https://api.opentopodata.org/v1/aster30m"
    params = {"locations": f"{lat},{lon}"}
    try:
        r = requests.get(url, params=params, timeout=5)
        if r.status_code == 200:
            return r.json()["results"][0]["elevation"]
    except:
        pass
    return None

def get_surrounding_elevations(lat, lon, radius_deg=0.01):
    points = {
        'center': (lat, lon),
        'north': (lat + radius_deg, lon),
        'south': (lat - radius_deg, lon),
        'east': (lat, lon + radius_deg),
        'west': (lat, lon - radius_deg),
    }
    elevs = {}
    for name, (la, lo) in points.items():
        e = get_elevation(la, lo)
        if e is not None:
            elevs[name] = e
    return elevs

print("Получение высот вокруг станции...")
elev_data = get_surrounding_elevations(LATITUDE, LONGITUDE)

if len(elev_data) >= 3:
    heights = list(elev_data.values())
    median_h = np.median(heights)
    range_h = np.max(heights) - np.min(heights)
    std_h = np.std(heights)
    center_h = elev_data.get('center', median_h)
    relative_h = center_h - median_h

    slope_steepness = 0.0
    slope_aspect = 0.0
    radius_deg = 0.01
    if 'north' in elev_data and 'south' in elev_data:
        d_ns = (elev_data['north'] - elev_data['south']) / (2 * radius_deg * 111000)
    else:
        d_ns = 0.0
    if 'east' in elev_data and 'west' in elev_data:
        d_ew = (elev_data['east'] - elev_data['west']) / (2 * radius_deg * 111000 * np.cos(np.deg2rad(LATITUDE)))
    else:
        d_ew = 0.0
    slope_steepness = np.sqrt(d_ns**2 + d_ew**2)
    slope_aspect = np.arctan2(d_ns, d_ew)

    if slope_steepness < 0.02:
        terrain_type = 0
    elif slope_steepness < 0.1:
        terrain_type = 1
    else:
        terrain_type = 2

    print(f"Рельеф: median_h={median_h:.1f}, range={range_h:.1f}, std={std_h:.1f}, "
          f"relative={relative_h:.1f}, slope={slope_steepness:.4f}, aspect={np.rad2deg(slope_aspect):.1f}°, terrain={terrain_type}")
else:
    median_h = 36.0; range_h = 0.0; std_h = 0.0; relative_h = 0.0
    slope_steepness = 0.0; slope_aspect = 0.0; terrain_type = 0
    print("Недостаточно данных рельефа, используем заглушки.")

# =============================================================================
# 6. ОБРАБОТКА NASA POWER
# =============================================================================
def process_power(df_power):
    ws10 = df_power['ws10_power'].values
    ws50 = df_power['ws50_power'].values
    ratio = ws50 / np.clip(ws10, 0.1, None)
    alpha = np.log(np.clip(ratio, 0.1, None)) / np.log(50.0 / 10.0)
    df_power['ws84_power'] = ws10 * (84.0 / 10.0) ** alpha

    temp_k = df_power['temp2m_power'].values + 273.15
    press_pa = df_power['pressure_power'].values * 100
    qv = df_power['qv2m_power'].values / 1000.0
    temp_v = temp_k * (1 + 0.61 * qv)
    df_power['air_density_power'] = press_pa / (287.05 * temp_v)

    ws84 = df_power['ws84_power'].values
    df_power['ws84_cubed_power'] = ws84 ** 3
    theory = np.zeros_like(ws84)
    reg = (ws84 >= 3.0) & (ws84 < 10.3)
    theory[reg] = 90.09 * ((ws84[reg] - 3.0) / (10.3 - 3.0)) ** 3
    rated = (ws84 >= 10.3) & (ws84 <= 25.0)
    theory[rated] = 90.09
    df_power['theory_power_total_power'] = theory

    for lag in [1, 2, 3]:
        df_power[f'ws84_lag{lag}_power'] = df_power['ws84_power'].shift(lag)
    return df_power

train_power = process_power(train_power)
valid_power = process_power(valid_power)

# =============================================================================
# 7. ОБРАБОТКА ERA5
# =============================================================================
def process_era5(df_era5):
    HUB_H, LOW_H, HIGH_H = 84.0, 10.0, 100.0
    ws10 = df_era5['ws10_era5'].values
    ws100 = df_era5['ws100_era5'].values
    ratio = ws100 / np.clip(ws10, 0.1, None)
    alpha = np.log(np.clip(ratio, 0.1, None)) / np.log(HIGH_H / LOW_H)
    df_era5['wind_speed_84m_era5'] = ws10 * (HUB_H / LOW_H) ** alpha

    wd10_rad = np.deg2rad(df_era5['wd10_era5'].values)
    wd100_rad = np.deg2rad(df_era5['wd100_era5'].values)
    sin10, cos10 = np.sin(wd10_rad), np.cos(wd10_rad)
    sin100, cos100 = np.sin(wd100_rad), np.cos(wd100_rad)
    w_low = (HIGH_H - HUB_H) / (HIGH_H - LOW_H)
    sin84 = sin10 * w_low + sin100 * (1 - w_low)
    cos84 = cos10 * w_low + cos100 * (1 - w_low)
    df_era5['wind_direction_84m_era5'] = np.arctan2(sin84, cos84) / (2 * np.pi) % 1.0

    df_era5['temperature_84m_era5'] = df_era5['temp2m_era5'].values - 0.65 * (84 - 2) / 100
    df_era5['pressure_msl_era5'] = df_era5['pressure_era5'].values

    ws84 = df_era5['wind_speed_84m_era5'].values
    df_era5['ws84_cubed_era5'] = ws84 ** 3
    wd84_rad = 2 * np.pi * df_era5['wind_direction_84m_era5'].values
    df_era5['u84_era5'] = ws84 * np.cos(wd84_rad)
    df_era5['v84_era5'] = ws84 * np.sin(wd84_rad)

    press = df_era5['pressure_msl_era5']
    temp84 = df_era5['temperature_84m_era5']
    df_era5['air_density_era5'] = press * 100 / (287.05 * (temp84 + 273.15))
    df_era5['wind_power_density_era5'] = 0.5 * df_era5['air_density_era5'] * df_era5['ws84_cubed_era5']

    theory = np.zeros_like(ws84)
    reg = (ws84 >= 3.0) & (ws84 < 10.3)
    theory[reg] = 90.09 * ((ws84[reg] - 3.0) / (10.3 - 3.0)) ** 3
    rated = (ws84 >= 10.3) & (ws84 <= 25.0)
    theory[rated] = 90.09
    df_era5['theory_power_total_era5'] = theory

    for lag in [1, 2, 3]:
        df_era5[f'ws84_lag{lag}_era5'] = df_era5['wind_speed_84m_era5'].shift(lag)
    wd_series = df_era5['wind_direction_84m_era5']
    for lag in [1, 2, 3]:
        wd_lag = wd_series.shift(lag)
        df_era5[f'wd84_lag{lag}_sin_era5'] = np.sin(2 * np.pi * wd_lag)
        df_era5[f'wd84_lag{lag}_cos_era5'] = np.cos(2 * np.pi * wd_lag)

    df_era5['shear_120_84_era5'] = ws100 - ws84
    df_era5['shear_84_10_era5'] = ws84 - ws10

    return df_era5

train_era5 = process_era5(train_era5)
valid_era5 = process_era5(valid_era5)

# =============================================================================
# 8. ИНТЕРПОЛЯЦИЯ ИСХОДНЫХ ДАННЫХ НА 84 м (градусы/1000 → радианы)
# =============================================================================
HUB_H, LOW_H, HIGH_H = 84.0, 80.0, 120.0
for df in [train, valid]:
    ratio = df['wind_speed_120m'] / df['wind_speed_80m'].clip(lower=0.1)
    alpha = np.log(ratio.clip(lower=0.1)) / np.log(HIGH_H / LOW_H)
    df['wind_speed_84m'] = df['wind_speed_80m'] * (HUB_H / LOW_H) ** alpha

    wd80_rad = np.deg2rad(df['wind_direction_80m'] * 1000)
    wd120_rad = np.deg2rad(df['wind_direction_120m'] * 1000)
    sin_low, cos_low = np.sin(wd80_rad), np.cos(wd80_rad)
    sin_high, cos_high = np.sin(wd120_rad), np.cos(wd120_rad)
    w = (HIGH_H - HUB_H) / (HIGH_H - LOW_H)
    sin84 = sin_low * w + sin_high * (1 - w)
    cos84 = cos_low * w + cos_high * (1 - w)
    df['wind_direction_84m'] = np.arctan2(sin84, cos84) / (2 * np.pi) % 1.0

    if 'temperature_120m' in df.columns:
        df['temperature_84m'] = df['temperature_80m'] * w + df['temperature_120m'] * (1 - w)
    else:
        df['temperature_84m'] = df['temperature_80m']

def add_base_features(df):
    df = df.copy()
    dt = df[DT_COL]
    df['dayofyear'] = dt.dt.dayofyear
    df['dayofweek'] = dt.dt.dayofweek
    df['week']      = dt.dt.isocalendar().week.astype(int)
    df['hour_sin']  = np.sin(2 * np.pi * df['hour_of_day'] / 24)
    df['hour_cos']  = np.cos(2 * np.pi * df['hour_of_day'] / 24)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['doy_sin']   = np.sin(2 * np.pi * df['dayofyear'] / 365)
    df['doy_cos']   = np.cos(2 * np.pi * df['dayofyear'] / 365)

    for col in ['wind_direction_10m', 'wind_direction_80m',
                'wind_direction_120m', 'wind_direction_180m']:
        rad = np.deg2rad(df[col] * 1000)
        df[col + '_sin'] = np.sin(rad)
        df[col + '_cos'] = np.cos(rad)

    df['wind_speed_80m_sq']    = df['wind_speed_80m'] ** 2
    df['wind_speed_80m_cube']  = df['wind_speed_80m'] ** 3
    df['wind_speed_120m_cube'] = df['wind_speed_120m'] ** 3
    df['wind_shear']           = df['wind_speed_120m'] - df['wind_speed_10m']
    df['gust_ratio']           = df['wind_gusts_10m'] / (df['wind_speed_10m'] + 0.1)
    df['wind_avg']             = df[['wind_speed_80m', 'wind_speed_120m']].mean(axis=1)

    df['ws84_cubed'] = df['wind_speed_84m'] ** 3
    wd = 2 * np.pi * df['wind_direction_84m']
    df['u84'] = df['wind_speed_84m'] * np.cos(wd)
    df['v84'] = df['wind_speed_84m'] * np.sin(wd)

    if 'pressure_msl' in df.columns and 'temperature_84m' in df.columns:
        df['air_density'] = df['pressure_msl'] * 100 / (287.05 * (df['temperature_84m'] + 273.15))
        df['wind_power_density'] = 0.5 * df['air_density'] * df['ws84_cubed']

    if 'wind_speed_120m' in df.columns:
        df['shear_120_84'] = df['wind_speed_120m'] - df['wind_speed_84m']
    if 'wind_speed_10m' in df.columns:
        df['shear_84_10'] = df['wind_speed_84m'] - df['wind_speed_10m']

    ws = df['wind_speed_84m'].values
    theory = np.zeros_like(ws)
    reg = (ws >= 3.0) & (ws < 10.3)
    theory[reg] = 90.09 * ((ws[reg] - 3.0) / (10.3 - 3.0))**3
    rated = (ws >= 10.3) & (ws <= 25.0)
    theory[rated] = 90.09
    df['theory_power_total'] = theory

    for lag in [1, 2, 3]:
        df[f'ws84_lag{lag}'] = df['wind_speed_84m'].shift(lag)
    for lag in [1, 2, 3]:
        wd_lag = df['wind_direction_84m'].shift(lag)
        df[f'wd84_lag{lag}_sin'] = np.sin(2 * np.pi * wd_lag)
        df[f'wd84_lag{lag}_cos'] = np.cos(2 * np.pi * wd_lag)

    # =========================================================================
    # ИНТЕРАКТИВНЫЕ ПРИЗНАКИ РЕЛЬЕФА
    # =========================================================================
    df['elevation_median'] = median_h
    df['elevation_range'] = range_h
    df['elevation_std'] = std_h
    df['relative_elevation'] = relative_h
    df['slope_steepness'] = slope_steepness
    df['slope_aspect_sin'] = np.sin(slope_aspect)
    df['slope_aspect_cos'] = np.cos(slope_aspect)
    df['terrain_type'] = terrain_type

    if slope_steepness > 0:
        df['wind_slope_proj'] = df['u84'] * np.sin(slope_aspect) + df['v84'] * np.cos(slope_aspect)
        df['wind_slope_cross'] = -df['u84'] * np.cos(slope_aspect) + df['v84'] * np.sin(slope_aspect)
        df['wind_up_slope'] = (df['wind_slope_proj'] > 0).astype(int)
        df['wind_speed_x_slope'] = df['wind_speed_84m'] * slope_steepness
    else:
        df['wind_slope_proj'] = 0.0
        df['wind_slope_cross'] = 0.0
        df['wind_up_slope'] = 0
        df['wind_speed_x_slope'] = 0.0

    df['repair_ma6']  = df['Кол-во_ВЭУ_в_ремонте'].rolling(6, min_periods=1).mean()
    df['has_repair']  = (df['Кол-во_ВЭУ_в_ремонте'] > 0).astype(int)
    df['turbines_available'] = N_TURBINES - df['Кол-во_ВЭУ_в_ремонте']
    df['available_ratio']    = df['turbines_available'] / N_TURBINES
    return df

def fill_missing_180m(df):
    df = df.copy()
    df['wind_speed_180m']         = df['wind_speed_180m'].fillna(df['wind_speed_120m'])
    df['wind_direction_180m']     = df['wind_direction_180m'].fillna(df['wind_direction_120m'])
    df['wind_direction_180m_sin'] = df['wind_direction_180m_sin'].fillna(df['wind_direction_120m_sin'])
    df['wind_direction_180m_cos'] = df['wind_direction_180m_cos'].fillna(df['wind_direction_120m_cos'])
    return df

train_f = fill_missing_180m(add_base_features(train))
valid_f = fill_missing_180m(add_base_features(valid))

# =============================================================================
# 9. ОБЪЕДИНЕНИЕ ВСЕХ ИСТОЧНИКОВ
# =============================================================================
lag_cols = [c for c in train_f.columns if 'lag' in c and not ('_era5' in c or '_power' in c)]
train_f = train_f.dropna(subset=lag_cols).reset_index(drop=True)
valid_f[lag_cols] = valid_f[lag_cols].fillna(train_f[lag_cols].median())

# Добавляем ERA5 и NASA POWER
for src_df in [train_era5, train_power]:
    src_df[DT_COL] = src_df[DT_COL].astype(train_f[DT_COL].dtype)
    train_f = train_f.merge(src_df, on=DT_COL, how='left')

for src_df in [valid_era5, valid_power]:
    src_df[DT_COL] = src_df[DT_COL].astype(valid_f[DT_COL].dtype)
    valid_f = valid_f.merge(src_df, on=DT_COL, how='left')

# Добавляем AOD
if not train_aod.empty:
    train_aod[DT_COL] = train_aod[DT_COL].astype(train_f[DT_COL].dtype)
    train_f = train_f.merge(train_aod, on=DT_COL, how='left')
    if 'aod' in train_f.columns:
        train_f['aod'] = train_f['aod'].fillna(train_f['aod'].median())
    else:
        train_f['aod'] = 0.0
else:
    train_f['aod'] = 0.0

if not valid_aod.empty:
    valid_aod[DT_COL] = valid_aod[DT_COL].astype(valid_f[DT_COL].dtype)
    valid_f = valid_f.merge(valid_aod, on=DT_COL, how='left')
    if 'aod' in valid_f.columns:
        # Заполняем пропуски медианой из обучающей выборки
        valid_f['aod'] = valid_f['aod'].fillna(train_f['aod'].median() if 'aod' in train_f.columns else 0.0)
    else:
        valid_f['aod'] = 0.0
else:
    valid_f['aod'] = 0.0

# Заполняем пропуски в ERA5 и POWER
for suffix in ['_era5', '_power']:
    cols = [c for c in train_f.columns if c.endswith(suffix)]
    train_f[cols] = train_f[cols].fillna(train_f[cols].median())
    valid_f[cols] = valid_f[cols].fillna(train_f[cols].median())

# =============================================================================
# 10. WAKE EFFECT – секторные коэффициенты недобора
# =============================================================================
dir_84 = train_f['wind_direction_84m']
sector = (dir_84 * N_WAKE_SECTORS).astype(int) % N_WAKE_SECTORS
fact_vs_theory = train_f[TARGET] / (train_f['theory_power_total'] + 1e-6)
wake_factors = fact_vs_theory.groupby(sector).mean()

train_f['wake_factor'] = (train_f['wind_direction_84m'] * N_WAKE_SECTORS).astype(int) % N_WAKE_SECTORS
train_f['wake_factor'] = train_f['wake_factor'].map(wake_factors)
valid_f['wake_factor'] = (valid_f['wind_direction_84m'] * N_WAKE_SECTORS).astype(int) % N_WAKE_SECTORS
valid_f['wake_factor'] = valid_f['wake_factor'].map(wake_factors)

train_f['wake_corrected_power'] = train_f['theory_power_total'] * train_f['wake_factor']
valid_f['wake_corrected_power'] = valid_f['theory_power_total'] * valid_f['wake_factor']

# =============================================================================
# 11. ФОРМИРУЕМ МАССИВЫ
# =============================================================================
EXCLUDE_FEATURES = [TARGET, DT_COL, 'Кол-во_ВЭУ_в_ремонте']
FEATURES = [c for c in train_f.columns if c not in EXCLUDE_FEATURES]
print(f"Финальное количество признаков: {len(FEATURES)}")

X = train_f[FEATURES].values
y = train_f[TARGET].values
X_valid = valid_f[FEATURES].values

# =============================================================================
# 12. ОПТИМИЗАЦИЯ LightGBM (Optuna) – КВАНТИЛЬНАЯ РЕГРЕССИЯ
# =============================================================================
def objective_lgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 2000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 31, 127),
        'max_depth': trial.suggest_int('max_depth', 5, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'objective': 'quantile',
        'alpha': 0.5,
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1
    }
    tscv = TimeSeriesSplit(n_splits=3)
    scores = []
    for tr_idx, val_idx in tscv.split(X):
        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]
        model = lgb.LGBMRegressor(**params)
        model.fit(X_tr, y_tr)
        preds = np.clip(model.predict(X_val), 0, CAPACITY)
        scores.append(mean_absolute_error(y_val, preds))
    return np.mean(scores)

print("Оптимизация LightGBM (квантильная)...")
study_lgb = optuna.create_study(direction='minimize')
study_lgb.optimize(objective_lgb, n_trials=30, show_progress_bar=True)
best_lgb = study_lgb.best_params
best_lgb.update({
    'objective': 'quantile',
    'alpha': 0.5,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
})
print(f"Лучшие параметры LightGBM: {best_lgb}")

# =============================================================================
# 13. HOLDOUT-ВАЛИДАЦИЯ (янв-март 2025) + ПОДБОР ВЕСОВ
# =============================================================================
holdout_mask = (train_f[DT_COL] >= '2025-01-01') & (train_f[DT_COL] < '2025-04-01')
X_tr, X_ho = X[~holdout_mask], X[holdout_mask]
y_tr, y_ho = y[~holdout_mask], y[holdout_mask]

# LightGBM бленд (квантильный)
lgb_models_holdout, lgb_holdout_preds = [], []
for seed in SEEDS_LGB:
    m = lgb.LGBMRegressor(**best_lgb)
    m.fit(X_tr, y_tr, eval_set=[(X_ho, y_ho)], callbacks=[lgb.early_stopping(75, verbose=False)])
    lgb_models_holdout.append(m)
    lgb_holdout_preds.append(np.clip(m.predict(X_ho), 0, CAPACITY))
lgb_ho = np.mean(lgb_holdout_preds, axis=0)

# RF (обычный)
rf = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
rf.fit(X_tr, y_tr)
rf_ho = np.clip(rf.predict(X_ho), 0, CAPACITY)

# CatBoost бленд (обычный)
cat_holdout_preds = []
for seed in SEEDS_CAT:
    m = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6, random_seed=seed, verbose=False)
    m.fit(X_tr, y_tr, eval_set=(X_ho, y_ho), early_stopping_rounds=30, verbose=False)
    cat_holdout_preds.append(np.clip(m.predict(X_ho), 0, CAPACITY))
cat_ho = np.mean(cat_holdout_preds, axis=0)

# Подбор весов
best_weights, best_mae = None, np.inf
for w1 in range(0, 21):
    for w2 in range(0, 21 - w1):
        w3 = 20 - w1 - w2
        weights = np.array([w1, w2, w3]) / 20.0
        mae = mean_absolute_error(y_ho, weights[0]*lgb_ho + weights[1]*rf_ho + weights[2]*cat_ho)
        if mae < best_mae:
            best_mae = mae
            best_weights = weights.copy()
print(f"Лучшие веса: LGBM={best_weights[0]:.2f}, RF={best_weights[1]:.2f}, CatBoost={best_weights[2]:.2f} | MAE holdout={best_mae:.3f} МВт")

# =============================================================================
# 14. ФИНАЛЬНОЕ ОБУЧЕНИЕ + ПРОГНОЗ
# =============================================================================
n_iter_ho = int(np.median([m.best_iteration_ or 1500 for m in lgb_models_holdout]) * LGB_ITER_MULTIPLIER)
best_lgb['n_estimators'] = n_iter_ho

# LightGBM финальный бленд
lgb_models = [lgb.LGBMRegressor(**best_lgb).fit(X, y) for seed in SEEDS_LGB]
lgb_valid = np.mean([np.clip(m.predict(X_valid), 0, CAPACITY) for m in lgb_models], axis=0)

# RF финальный
rf_final = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
rf_final.fit(X, y)
rf_valid = np.clip(rf_final.predict(X_valid), 0, CAPACITY)

# CatBoost финальный бленд
cat_models = []
for seed in SEEDS_CAT:
    m = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6, random_seed=seed, verbose=False)
    m.fit(X, y, verbose=False)
    cat_models.append(m)
cat_valid = np.mean([np.clip(m.predict(X_valid), 0, CAPACITY) for m in cat_models], axis=0)

final_pred = (best_weights[0]*lgb_valid + best_weights[1]*rf_valid + best_weights[2]*cat_valid)

zero_mask = valid_f['turbines_available'].values <= 0
final_pred[zero_mask] = 0.0
final_pred = np.minimum(final_pred, CAPACITY * valid_f['available_ratio'].values + 1e-3)
final_pred = np.clip(final_pred, 0, CAPACITY)

result = (pd.DataFrame({DT_COL: valid_f[DT_COL].values, TARGET: final_pred})
            .set_index(DT_COL).loc[valid_order].reset_index())
result[[TARGET]].to_csv(OUTPUT_VALID, index=False)
print(f"\n[OK] {OUTPUT_VALID} сохранён ({len(result)} строк)")